In [13]:
from google.colab import files

uploaded = files.upload()

Saving Online Retail.csv to Online Retail (1).csv


In [14]:
import pandas as pd
import numpy as np

df = pd.read_csv("Online Retail.csv", encoding='latin1')
df = df.rename(columns={
    "InvoiceNo": "invoice_no",
    "StockCode": "stock_code",
    "Description": "description",
    "Quantity": "quantity",
    "InvoiceDate": "invoice_date",
    "UnitPrice": "unit_price",
    "CustomerID": "customer_id",
    "Country": "country"
})
print(df.columns.tolist())
print()

['invoice_no', 'stock_code', 'description', 'quantity', 'invoice_date', 'unit_price', 'customer_id', 'country']



upload, read file. rename columns

In [15]:
df = df.assign(
    invoice_date=pd.to_datetime(df["invoice_date"]),
    description=df["description"].astype(str).str.strip(),
    customer_id=df["customer_id"].fillna(0).astype(int),
    line_total=df["quantity"] * df["unit_price"],
    order_type=np.where(df["invoice_no"].astype(str).str.startswith("C"), "Cancelled", "Normal")
)
print(df[["invoice_no", "description", "quantity", "unit_price", "line_total", "order_type"]].head())
print()

  invoice_no                          description  quantity  unit_price  \
0     536365   WHITE HANGING HEART T-LIGHT HOLDER         6        2.55   
1     536365                  WHITE METAL LANTERN         6        3.39   
2     536365       CREAM CUPID HEARTS COAT HANGER         8        2.75   
3     536365  KNITTED UNION FLAG HOT WATER BOTTLE         6        3.39   
4     536365       RED WOOLLY HOTTIE WHITE HEART.         6        3.39   

   line_total order_type  
0       15.30     Normal  
1       20.34     Normal  
2       22.00     Normal  
3       20.34     Normal  
4       20.34     Normal  



clean columns and add new varaibles

In [16]:
df_clean = df.dropna(subset=["description"]).copy()

clean up blank spaces

In [17]:
df_long = pd.melt(
    df_clean,
    id_vars=["invoice_no", "stock_code", "description", "invoice_date", "customer_id", "country", "order_type"],
    value_vars=["quantity", "unit_price", "line_total"],
    var_name="metric",
    value_name="value"
)

print("Long / tidy version")
print(df_long.head(10))
print()


Long / tidy version
  invoice_no stock_code                          description  \
0     536365     85123A   WHITE HANGING HEART T-LIGHT HOLDER   
1     536365      71053                  WHITE METAL LANTERN   
2     536365     84406B       CREAM CUPID HEARTS COAT HANGER   
3     536365     84029G  KNITTED UNION FLAG HOT WATER BOTTLE   
4     536365     84029E       RED WOOLLY HOTTIE WHITE HEART.   
5     536365      22752         SET 7 BABUSHKA NESTING BOXES   
6     536365      21730    GLASS STAR FROSTED T-LIGHT HOLDER   
7     536366      22633               HAND WARMER UNION JACK   
8     536366      22632            HAND WARMER RED POLKA DOT   
9     536367      84879        ASSORTED COLOUR BIRD ORNAMENT   

         invoice_date  customer_id         country order_type    metric  value  
0 2010-12-01 08:26:00        17850  United Kingdom     Normal  quantity    6.0  
1 2010-12-01 08:26:00        17850  United Kingdom     Normal  quantity    6.0  
2 2010-12-01 08:26:00        178

reshape to long version

In [18]:
country_summary = (
    df_clean.groupby("country")[["line_total"]]
    .sum()
    .sort_values("line_total", ascending=False)
)

print("Total sales by country")
print(country_summary.head(10))
print()

Total sales by country
                 line_total
country                    
United Kingdom  8187806.364
Netherlands      284661.540
EIRE             263276.820
Germany          221698.210
France           197403.900
Australia        137077.270
Switzerland       56385.350
Spain             54774.580
Belgium           40910.960
Sweden            36595.910



group by country and display sum sales

In [20]:
country_order_summary = pd.pivot_table(
    df_clean,
    index="country",
    columns="order_type",
    values="line_total",
    aggfunc="sum",
    fill_value=0
)

print("Pivot table by country and order type")
print(country_order_summary.head(10))

Pivot table by country and order type
order_type       Cancelled     Normal
country                              
Australia         -1444.04  138521.31
Austria             -44.36   10198.68
Bahrain            -205.74     754.14
Belgium            -285.38   41196.34
Brazil                0.00    1143.60
Canada                0.00    3666.38
Channel Islands    -364.15   20450.44
Cyprus             -644.09   13590.38
Czech Republic     -119.02     826.74
Denmark            -187.20   18955.34


create table with columns. print tidy data.